In [280]:
#-- Packages --#

#--- Operational ---#
import os
import sys 
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
import json
import re
import geopandas as gpd
from __future__ import annotations
from typing import Any, Mapping, Sequence


#--- Visualisations ---#
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import seaborn as sns

#-- Directories --#
nb_dir = Path.cwd()
REPO_ROOT = nb_dir.parent

data_dir = REPO_ROOT / 'data/'
docs_dir = REPO_ROOT / 'docs/'

analysis_dir = data_dir / 'processed' / 'analysis/'
fires_dir = analysis_dir / "national_canadian_fires"

cache_dir = data_dir / 'processed' / 'cached/'

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

#--- Constants ---#
cached_fires = cache_dir / 'NBAC/Canada_fires_1990_2024.parquet'

In [281]:
#-- Helper Functions --#
def extract_max_year(path):
    """
    Identify shapefile with latest year of available data
    """
    years = re.findall(r"\d{4}", path.stem)
    return max(map(int, years)) if years else -1


In [282]:
# #-- Load Files --#

# # National Canada polygons (GeoJSON)
# print(f'Loading National fires GeoJSON...')
# fires_GeoJSON_path = analysis_dir / "national_canadian_fires/Canada_fires_1990_2024.geojson"
# fires_GeoJSON = gpd.read_file(fires_GeoJSON_path)
# print(f" National Canada fires GeoJSON loaded. {fires_GeoJSON.crs}\n")

# # National Canada polygons (Shapefile)

# shp_files = list(fires_dir.glob("*.shp"))

# if not shp_files:
#     raise FileNotFoundError(f"No shapefiles found in {fires_dir}\n")

# fires_path = max(shp_files, key=extract_max_year)

# print(f"Loading National fires shapefile... \n File name: {fires_path.name}")
# fire_stats = gpd.read_file(fires_path)
# print(f" National Canada Fires loaded. {fire_stats.crs}\n")

# print(f"Cache National fires shapefile as parquet")
# cached_path = cache_dir / f"{fires_path.name[:-4]}.parquet"
# try:
#     print(" Caching...")
#     fire_stats.to_parquet(cached_path, index=False)
#     print(f" Cache complete. \n Saved to: {cached_path}" )
# except Exception as e:
#     raise RuntimeError(f"National fires shapefile failed to export as parquet: {e}")

In [283]:

# Load cached files

if not cached_fires.is_file():
    raise FileNotFoundError(
        f"NBAC cached parquet not found: {cached_fires}\n"
        "Run the 'Load File' block to generate it."
    )

try:
    NBAC = gpd.read_parquet(cached_fires)
    print("NBAC Wildfires cached parquet file found.")
    print(" Loading cached fires...")
    print(f" NBAC fires loaded. CRS: {NBAC.crs}\n")
except Exception as e:
    raise RuntimeError(
        f"Failed to read cached parquet: {cached_fires}\n"
        f"Error: {e}"
    ) from e


# Load AvCan regions
# =================================================================

print(f'Loading AvCan regions shapefile...')
avcan_path = analysis_dir / "avalanche_canada/regions/AvCan_cleaned_subregions.geojson"
avcan_regions= gpd.read_file(avcan_path)
print(f" AvCan Regions loaded. CRS: {avcan_regions.crs}\n")

# AvCan Fires 
# =================================================================

print(f'Loading all AvCan fires shapefile...')
avcan_fires_file = analysis_dir / 'avalanche_canada/fires/AvCan_fires_1990_2024.shp'

avcan_fires = gpd.read_file(avcan_fires_file)
print(f' AvCan fires loaded. CRS: {avcan_fires.crs}\n')



NBAC Wildfires cached parquet file found.
 Loading cached fires...
 NBAC fires loaded. CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longi

## Cleaning

In [284]:

RENAME = {

    # Shared columns
    "gid": "Unique_gid_ID",
    "fireid": "Fire ID",
    "year": "Year",
    "prov_terr" : "Province/Territory",
    "natpark": "National Park",
    "cause": "Cause",

    # NBAC columns
    "adj_ha": "Adjusted Burn Area (ha)",

    # AvCan columns
    "region":"Region",
    "subregion":"Subregion",
    "subreg_ha":"Subregion Area (ha)",
    "tot_adj_ha":"Total Adjusted Area (ha)"

}

NBAC_stats = NBAC.rename(columns=RENAME).copy()
avcan_stats = avcan_fires.rename(columns=RENAME).copy()

In [285]:
NBAC_stats.columns

Index(['Unique_gid_ID', 'Fire ID', 'Year', 'Province/Territory',
       'National Park', 'Adjusted Burn Area (ha)', 'Cause', 'geometry'],
      dtype='object')

### Column Creation

In [286]:
NBAC_stats["Adjusted Burn Area (ha)"] = pd.to_numeric(NBAC_stats["Adjusted Burn Area (ha)"], errors="coerce")
avcan_stats["Total Adjusted Area (ha)"] = pd.to_numeric(avcan_stats["Total Adjusted Area (ha)"], errors="coerce")



NBAC_cause = NBAC_stats["Cause"].astype(str).str.strip().str.lower()
NBAC_stats["Is_Natural"] = (NBAC_cause == "natural").astype("int64")
NBAC_stats["Is_Human"] = (NBAC_cause == "human").astype("int64")
NBAC_stats["Is_Undetermined"] = (NBAC_cause == "undetermined").astype("int64")


avcan_cause = avcan_stats["Cause"].astype(str).str.strip().str.lower()
avcan_stats["Is_Natural"] = (avcan_cause == "natural").astype("int64")
avcan_stats["Is_Human"] = (avcan_cause == "human").astype("int64")
avcan_stats["Is_Undetermined"] = (avcan_cause == "undetermined").astype("int64")

In [287]:



area_yearly = (
    NBAC_stats
      .groupby(["Province/Territory", "Year"], dropna=False)
      .agg(
          fires_n=("Unique_gid_ID", "nunique"),        # unique fires
          rows_n=("Unique_gid_ID", "size"),            # number of records (optional)
          area_sum_ha=("Adjusted Burn Area (ha)", "sum"),     # total burned area
          area_mean_ha=("Adjusted Burn Area (ha)", "mean"),   # mean per-record area (see note below)
          natural_cause=("Is_Natural", "sum"),
          human_cause=("Is_Human", "sum"),
          undetermined_cause=("Is_Undetermined", "sum")
      )
      .reset_index()
)

nbac_yearly = (
    NBAC_stats
      .groupby("Year", dropna=False)
      .agg(
          fires_n=("Unique_gid_ID", "nunique"),        # unique fires
          rows_n=("Unique_gid_ID", "size"),            # number of records (optional)
          area_sum_ha=("Adjusted Burn Area (ha)", "sum"),     # total burned area
          area_mean_ha=("Adjusted Burn Area (ha)", "mean"),   # mean per-record area (see note below)
          natural_cause=("Is_Natural", "sum"),
          human_cause=("Is_Human", "sum"),
          undetermined_cause=("Is_Undetermined", "sum")
      )
      .reset_index()
)

nbac_yearly["natural_pct"] = (nbac_yearly["natural_cause"]/ nbac_yearly['fires_n'] ) * 100

nbac_yearly["human_pct"] = (nbac_yearly["human_cause"]/ nbac_yearly['fires_n']) * 100

nbac_yearly["undetermined_pct"] = (nbac_yearly["undetermined_cause"]/ nbac_yearly['fires_n'] ) * 100


In [288]:
den = area_yearly["rows_n"].replace(0, pd.NA)

area_yearly["natural_pct"] = (area_yearly["natural_cause"] / den) * 100
area_yearly["human_pct"] = (area_yearly["human_cause"] / den) * 100
area_yearly["undetermined_pct"] = (area_yearly["undetermined_cause"] / den) * 100


In [289]:
bc_yearly = area_yearly[area_yearly["Province/Territory"] == "BC"].copy()


## Visualisations

### Canada

In [290]:
fig = go.Figure()

fig.update_layout(
    title_text = "Canadian Wildfires and Causes Over Time",
    xaxis_title = "Years",
    yaxis_title = "Count"
)

fig.add_trace( go.Scatter(
    x = nbac_yearly["Year"],
    y = nbac_yearly["fires_n"],
    name = "Canadian Wildfires",
    showlegend = True,

    customdata = (np.column_stack([
        nbac_yearly["natural_pct"],
        nbac_yearly["human_pct"],
        nbac_yearly["undetermined_pct"]
    ])),

    hovertemplate = (
        "Year: %{x}"
        "<br>Count: %{y}"
        "<br>Natural Cause: %{customdata[0]:.2f} %"
        "<br>Human Cause: %{customdata[1]:.2f} %"
        "<br>Undetermined Cause: %{customdata[2]:.2f} %"
    )

))

fig.add_trace(go.Scatter(
    x = nbac_yearly["Year"],
    y = nbac_yearly['natural_cause'],
    name = "Natural Cause",

    customdata = (
        nbac_yearly["natural_pct"]),


    hovertemplate = (
        "Year: %{x}"
        "<br>Count: %{y}"
        "<br>Natural Cause: %{customdata:.2f} %"
    )
    
))

fig.add_trace(go.Scatter(
    x = nbac_yearly["Year"],
    y = nbac_yearly["human_cause"],
    name = "Human Cause",
    
    customdata = (
        nbac_yearly["human_pct"]),


    hovertemplate = (
        "Year: %{x}"
        "<br>Count: %{y}"
        "<br>Human Cause: %{customdata:.2f} %"
    )
))

fig.add_trace(go.Scatter(
    x = nbac_yearly["Year"],
    y = nbac_yearly["undetermined_cause"],
    name = "Undetermined Cause",
    
    customdata = (
        nbac_yearly["undetermined_pct"]),


    hovertemplate = (
        "Year: %{x}"
        "<br>Count: %{y}"
        "<br>Undetermined Cause: %{customdata:.2f} %"
    )
))
fig.show()

In [291]:
nbac_yearly.head()

,Year,fires_n,rows_n,area_sum_ha,area_mean_ha,natural_cause,human_cause,undetermined_cause,natural_pct,human_pct,undetermined_pct
0,1990,509,512,8.587488e+05,1677.243712,229,94,189,44.990177,18.467583,37.131631
1,1991,575,582,1.530291e+06,2629.365928,203,154,225,35.304348,26.782609,39.130435
2,1992,331,333,8.640520e+05,2594.750768,137,82,114,41.389728,24.773414,34.441088
3,1993,377,378,1.947867e+06,5153.086565,164,56,158,43.501326,14.854111,41.909814
4,1994,708,720,5.073875e+06,7047.048654,333,78,309,47.033898,11.016949,43.644068


In [292]:
fig0 = go.Figure()

fig0.update_layout(
    title_text = "Canadian Wildfires Burned Area Over Time",
    xaxis_title = "Years",
    yaxis_title = "Area (ha)"
)

fig0.add_trace( go.Scatter(
    x = nbac_yearly["Year"],
    y = nbac_yearly["area_sum_ha"],
    name = "Total Burn Area",
    showlegend = True,

    # customdata = (np.column_stack([
    # ])),

    hovertemplate = (
        "Year: %{x}"
        "<br>Area: %{y}"
    )
))

fig0.add_trace(go.Scatter(
    x = nbac_yearly["Year"],
    y = nbac_yearly["area_mean_ha"],
    name = "Mean Fire Burn Area",

    hovertemplate= (
        "Year: %{x}"
        "<br>Area %{y}"
    )
))


### British Columbia

In [293]:

fig1= go.Figure()

fig1.update_layout(
    title_text = "British Columbia Wildfires and Causes Over Time",
    xaxis_title = "Years",
    yaxis_title = "Count"
)

fig1.add_trace( go.Scatter(
    x = bc_yearly["Year"],
    y = bc_yearly["fires_n"],
    name = "British Columbia Wildfires",
    showlegend = True,

    customdata = (np.column_stack([
        bc_yearly["natural_pct"],
        bc_yearly["human_pct"],
        bc_yearly["undetermined_pct"]
    ])),

    hovertemplate = (
        "Year: %{x}"
        "<br>Count: %{y}"
        "<br>Natural Cause: %{customdata[0]:.2f} %"
        "<br>Human Cause: %{customdata[1]:.2f} %"
        "<br>Undetermined Cause: %{customdata[2]:.2f} %"
    )

))

fig1.add_trace(go.Scatter(
    x = bc_yearly["Year"],
    y = bc_yearly['natural_cause'],
    name = "Natural Cause",

    customdata = (
        bc_yearly["natural_pct"]),


    hovertemplate = (
        "Year: %{x}"
        "<br>Count: %{y}"
        "<br>Natural Cause: %{customdata:.2f} %"
    )
    
))

fig1.add_trace(go.Scatter(
    x = bc_yearly["Year"],
    y = bc_yearly["human_cause"],
    name = "Human Cause",
    
    customdata = (
        bc_yearly["human_pct"]),


    hovertemplate = (
        "Year: %{x}"
        "<br>Count: %{y}"
        "<br>Human Cause: %{customdata:.2f} %"
    )
))

fig1.add_trace(go.Scatter(
    x = bc_yearly["Year"],
    y = bc_yearly["undetermined_cause"],
    name = "Undetermined Cause",
    
    customdata = (
        bc_yearly["undetermined_pct"]),


    hovertemplate = (
        "Year: %{x}"
        "<br>Count: %{y}"
        "<br>Undetermined Cause: %{customdata:.2f} %"
    )
))
fig1.show()

In [294]:
fig2 = go.Figure()

fig2.update_layout(
    title_text = "British Columbia Wildfires Burned Area Over Time",
    xaxis_title = "Years",
    yaxis_title = "Area (ha)"
)

fig2.add_trace( go.Scatter(
    x = bc_yearly["Year"],
    y = bc_yearly["area_sum_ha"],
    name = "Total Burn Area",
    showlegend = True,

    # customdata = (np.column_stack([
    # ])),

    hovertemplate = (
        "Year: %{x}"
        "<br>Area: %{y}"
    )
))

fig2.add_trace(go.Scatter(
    x = bc_yearly["Year"],
    y = bc_yearly["area_mean_ha"],
    name = "Mean Fire Burn Area",

    hovertemplate= (
        "Year: %{x}"
        "<br>Area %{y}"
    )
))


### Avalanche Canada

In [296]:

avcan_yearly = (
    avcan_stats
      .groupby("Year", dropna=False)
      .agg(
          fires_n=("Unique_gid_ID", "nunique"),        # unique fires
          rows_n=("Unique_gid_ID", "size"),            # number of records (optional)
          area_sum_ha=("Total Adjusted Area (ha)", "sum"),     # total burned area
          area_mean_ha=("Total Adjusted Area (ha)", "mean"),   # mean per-record area (see note below)
          natural_cause=("Is_Natural", "sum"),
          human_cause=("Is_Human", "sum"),
          undetermined_cause=("Is_Undetermined", "sum")
      )
      .reset_index()
)

avcan_yearly["natural_pct"] = (avcan_yearly["natural_cause"]/ avcan_yearly['fires_n'] ) * 100

avcan_yearly["human_pct"] = (avcan_yearly["human_cause"]/ avcan_yearly['fires_n']) * 100

avcan_yearly["undetermined_pct"] = (avcan_yearly["undetermined_cause"]/ avcan_yearly['fires_n'] ) * 100

In [297]:
avcan_yearly.head()

,Year,fires_n,rows_n,area_sum_ha,area_mean_ha,natural_cause,human_cause,undetermined_cause,natural_pct,human_pct,undetermined_pct
0,1990,52,53,8427.238571,159.004501,33,15,5,63.461538,28.846154,9.615385
1,1991,25,27,2256.738729,83.582916,4,23,0,16.000000,92.000000,0.000000
2,1992,46,46,8943.788935,194.430194,23,19,4,50.000000,41.304348,8.695652
3,1993,16,16,1212.621280,75.788830,3,12,1,18.750000,75.000000,6.250000
4,1994,71,72,10801.197910,150.016638,43,25,4,60.563380,35.211268,5.633803


In [298]:

fig3= go.Figure()

fig3.update_layout(
    title_text = "AvCan Wildfires and Causes Over Time",
    xaxis_title = "Years",
    yaxis_title = "Count"
)

fig3.add_trace( go.Scatter(
    x = avcan_yearly["Year"],
    y = avcan_yearly["fires_n"],
    name = "AvCan Wildfires",
    showlegend = True,

    customdata = (np.column_stack([
        avcan_yearly["natural_pct"],
        avcan_yearly["human_pct"],
        avcan_yearly["undetermined_pct"]
    ])),

    hovertemplate = (
        "Year: %{x}"
        "<br>Count: %{y}"
        "<br>Natural Cause: %{customdata[0]:.2f} %"
        "<br>Human Cause: %{customdata[1]:.2f} %"
        "<br>Undetermined Cause: %{customdata[2]:.2f} %"
    )

))

fig3.add_trace(go.Scatter(
    x = avcan_yearly["Year"],
    y = avcan_yearly['natural_cause'],
    name = "Natural Cause",

    customdata = (
        avcan_yearly["natural_pct"]),


    hovertemplate = (
        "Year: %{x}"
        "<br>Count: %{y}"
        "<br>Natural Cause: %{customdata:.2f} %"
    )
    
))

fig3.add_trace(go.Scatter(
    x = avcan_yearly["Year"],
    y = avcan_yearly["human_cause"],
    name = "Human Cause",
    
    customdata = (
        avcan_yearly["human_pct"]),


    hovertemplate = (
        "Year: %{x}"
        "<br>Count: %{y}"
        "<br>Human Cause: %{customdata:.2f} %"
    )
))

fig3.add_trace(go.Scatter(
    x = avcan_yearly["Year"],
    y = avcan_yearly["undetermined_cause"],
    name = "Undetermined Cause",
    
    customdata = (
        avcan_yearly["undetermined_pct"]),


    hovertemplate = (
        "Year: %{x}"
        "<br>Count: %{y}"
        "<br>Undetermined Cause: %{customdata:.2f} %"
    )
))
fig3.show()

In [299]:
fig4 = go.Figure()

fig4.update_layout(
    title_text = "AvCan Wildfires Burned Area Over Time",
    xaxis_title = "Years",
    yaxis_title = "Area (ha)"
)

fig4.add_trace( go.Scatter(
    x = avcan_yearly["Year"],
    y = avcan_yearly["area_sum_ha"],
    name = "Total Burn Area",
    showlegend = True,

    # customdata = (np.column_stack([
    # ])),

    hovertemplate = (
        "Year: %{x}"
        "<br>Area: %{y}"
    )
))

fig4.add_trace(go.Scatter(
    x = avcan_yearly["Year"],
    y = avcan_yearly["area_mean_ha"],
    name = "Mean Fire Burn Area",

    hovertemplate= (
        "Year: %{x}"
        "<br>Area %{y}"
    )
))


### Combined

In [304]:
fig6 = go.Figure()

fig6.update_layout(
    title_text = "Canadian Wildfires Burned Area Over Time",
    xaxis_title = "Years",
    yaxis_title = "Area (ha)"
)

fig6.add_trace( go.Scatter(
    x = nbac_yearly["Year"],
    y = nbac_yearly["area_sum_ha"],
    name = "Canada Burn Area",
    showlegend = True,

    # customdata = (np.column_stack([
    # ])),

    hovertemplate = (
        "Year: %{x}"
        "<br>Area: %{y}"
    )
))

fig6.add_trace( go.Scatter(
    x = bc_yearly["Year"],
    y = bc_yearly["area_sum_ha"],
    name = "BC Burn Area",
    showlegend = True,

    # customdata = (np.column_stack([
    # ])),

    hovertemplate = (
        "Year: %{x}"
        "<br>Area: %{y}"
    )
))

fig6.add_trace( go.Scatter(
    x = avcan_yearly["Year"],
    y = avcan_yearly["area_sum_ha"],
    name = "AvCan Burn Area",
    showlegend = True,

    # customdata = (np.column_stack([
    # ])),

    hovertemplate = (
        "Year: %{x}"
        "<br>Area: %{y}"
    )
))
